# Model 3 V2 — Group-Aware Training

This notebook trains a candidate without mixing source-file groups between training and the primary holdout.

Important:
- `network_threat_model_experiment.pkl` is preserved.
- production artifacts are preserved.
- binary model is trained on source groups.
- family model is trained only on attack rows in training groups.
- class weights are derived from training data only.
- the selected holdout files are never used for fitting.

The family model is evaluated separately for **known families** (families present in training) and **unseen families** (families absent from training). Unseen-family classification is not treated as a model failure because a supervised classifier cannot learn a label that was absent from training.

In [1]:
from pathlib import Path
import json, warnings, os
import numpy as np
import pandas as pd
import joblib
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_sample_weight

warnings.filterwarnings("ignore")
PROJECT_DIR=Path(r"C:/Users/bimal/OneDrive/Desktop/EVIDENT")
RAW_DIR=PROJECT_DIR/"data/raw/behavior/cicids2017"
OUT_DIR=PROJECT_DIR/"models/behavior/experiments"
META_DIR=PROJECT_DIR/"models/behavior/metadata"
OUT_DIR.mkdir(parents=True,exist_ok=True); META_DIR.mkdir(parents=True,exist_ok=True)
RANDOM_STATE=42
HOLDOUT=[
"Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv",
"Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv",
"Friday-WorkingHours-Morning.pcap_ISCX.csv",
"Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv",
]
print("Holdout groups:")
for x in HOLDOUT: print(" -",x)

Holdout groups:
 - Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
 - Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
 - Friday-WorkingHours-Morning.pcap_ISCX.csv
 - Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv


In [3]:
FAMILY_MAP={
"BENIGN":"BENIGN","DDoS":"DOS_DDOS","DoS Hulk":"DOS_DDOS","DoS GoldenEye":"DOS_DDOS",
"DoS slowloris":"DOS_DDOS","DoS Slowhttptest":"DOS_DDOS","PortScan":"PORT_SCAN",
"FTP-Patator":"BRUTE_FORCE","SSH-Patator":"BRUTE_FORCE",
"Web Attack – Brute Force":"WEB_ATTACK","Web Attack – XSS":"WEB_ATTACK","Web Attack – Sql Injection":"WEB_ATTACK",
"Bot":"BOTNET","Infiltration":"INFILTRATION","Heartbleed":"HEARTBLEED"
}
def label_to_family(x):
    s=str(x).strip()
    if s in FAMILY_MAP: return FAMILY_MAP[s]
    sl=s.lower()
    if "web attack" in sl: return "WEB_ATTACK"
    if "patator" in sl: return "BRUTE_FORCE"
    if "dos" in sl or "ddos" in sl: return "DOS_DDOS"
    if "portscan" in sl: return "PORT_SCAN"
    if "infiltration" in sl: return "INFILTRATION"
    if "heartbleed" in sl: return "HEARTBLEED"
    if "bot" in sl: return "BOTNET"
    if sl=="benign": return "BENIGN"
    return "OTHER_ATTACK"

def label_col(df):
    return next(c for c in df.columns if str(c).strip().lower()=="label")

In [7]:
# ============================================================
# MODEL 3 V2 — GROUP-AWARE TRAINING DATA PREPARATION
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. PROJECT PATHS
# ------------------------------------------------------------

PROJECT_DIR = Path(r"C:\Users\bimal\OneDrive\Desktop\EVIDENT")

RAW_DIR = (
    PROJECT_DIR
    / "data"
    / "raw"
    / "behaviour"
)

print("Project directory:")
print(PROJECT_DIR)

print("\nRaw behaviour directory:")
print(RAW_DIR)

# ------------------------------------------------------------
# 2. VERIFY DIRECTORY
# ------------------------------------------------------------

if not PROJECT_DIR.exists():
    raise FileNotFoundError(
        f"Project directory does not exist:\n{PROJECT_DIR}"
    )

if not RAW_DIR.exists():
    raise FileNotFoundError(
        f"Behaviour dataset directory does not exist:\n{RAW_DIR}"
    )

if not RAW_DIR.is_dir():
    raise NotADirectoryError(
        f"Expected a directory but found:\n{RAW_DIR}"
    )

# ------------------------------------------------------------
# 3. DISCOVER CSV FILES
# ------------------------------------------------------------

CSV_FILES = sorted(RAW_DIR.glob("*.csv"))

print("\nCSV files discovered:", len(CSV_FILES))

for path in CSV_FILES:
    print(" -", path.name)

if not CSV_FILES:
    raise RuntimeError(
        f"No CSV files were found in:\n{RAW_DIR}\n\n"
        "Check that the CIC-IDS2017 CSV files are inside this folder."
    )

# ------------------------------------------------------------
# 4. GROUPS THAT MUST REMAIN COMPLETELY UNSEEN
# ------------------------------------------------------------

HOLDOUT = {
    "Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv",
    "Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv",
    "Friday-WorkingHours-Morning.pcap_ISCX.csv",
    "Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv",
}

print("\nGroup-aware holdouts:")

for name in sorted(HOLDOUT):
    print(" -", name)

# ------------------------------------------------------------
# 5. VERIFY HOLDOUT FILES EXIST
# ------------------------------------------------------------

available_names = {p.name for p in CSV_FILES}

missing_holdouts = HOLDOUT - available_names

if missing_holdouts:
    raise FileNotFoundError(
        "The following holdout files were not found:\n"
        + "\n".join(f" - {x}" for x in sorted(missing_holdouts))
    )

# ------------------------------------------------------------
# 6. TRAINING FILES
# ------------------------------------------------------------

TRAIN_FILES = [
    path for path in CSV_FILES
    if path.name not in HOLDOUT
]

print("\nTraining groups:")

for path in TRAIN_FILES:
    print(" -", path.name)

print("\nNumber of training groups:", len(TRAIN_FILES))

if not TRAIN_FILES:
    raise RuntimeError(
        "No training files remain after removing holdout groups."
    )

# ------------------------------------------------------------
# 7. LABEL COLUMN DETECTION
# ------------------------------------------------------------

def label_col(df):
    """
    Locate the CIC-IDS2017 target column robustly.
    """
    for col in df.columns:
        if str(col).strip().lower() == "label":
            return col

    raise ValueError(
        "Could not find the 'Label' column.\n"
        f"Available columns include:\n{list(df.columns[-10:])}"
    )


# ------------------------------------------------------------
# 8. CIC-IDS2017 LABEL -> ATTACK FAMILY
# ------------------------------------------------------------

def label_to_family(label):
    """
    Convert original CIC-IDS2017 labels into the unified
    Model 3 attack-family taxonomy.
    """

    if pd.isna(label):
        return "OTHER_ATTACK"

    s = str(label).strip().lower()

    # -------------------------
    # BENIGN
    # -------------------------
    if s == "benign":
        return "BENIGN"

    # -------------------------
    # HEARTBLEED
    # -------------------------
    if "heartbleed" in s:
        return "HEARTBLEED"

    # -------------------------
    # INFILTRATION
    # -------------------------
    if "infiltration" in s:
        return "INFILTRATION"

    # -------------------------
    # PORT SCAN
    # -------------------------
    if "portscan" in s or "port scan" in s:
        return "PORT_SCAN"

    # -------------------------
    # BOTNET
    # -------------------------
    if (
        "bot" in s
        or "botnet" in s
    ):
        return "BOTNET"

    # -------------------------
    # BRUTE FORCE
    # -------------------------
    if (
        "ftp-patator" in s
        or "ftp-patator" in s
        or "ssh-patator" in s
        or "brute force" in s
        or "bruteforce" in s
    ):
        return "BRUTE_FORCE"

    # CIC-IDS2017 commonly uses:
    # FTP-Patator
    # SSH-Patator
    if "patator" in s:
        return "BRUTE_FORCE"

    # -------------------------
    # WEB ATTACK
    # -------------------------
    if (
        "web attack" in s
        or "xss" in s
        or "sql injection" in s
        or "sql" in s and "injection" in s
    ):
        return "WEB_ATTACK"

    # -------------------------
    # DOS / DDOS
    # -------------------------
    if (
        "ddos" in s
        or s.startswith("dos")
        or "dos " in s
        or "slowhttptest" in s
        or "slowloris" in s
        or "goldeneye" in s
        or "hulk" in s
    ):
        return "DOS_DDOS"

    # -------------------------
    # FALLBACK
    # -------------------------
    return "OTHER_ATTACK"


# ------------------------------------------------------------
# 9. DETERMINISTIC TRAINING SETTINGS
# ------------------------------------------------------------

RANDOM_STATE = 42

# Maximum rows per source file.
#
# We do NOT randomly remove rare attack classes.
# Rare labels are preserved first.
TRAIN_PER_FILE_MAX = 100000

train_parts = []

# ------------------------------------------------------------
# 10. LOAD TRAINING GROUPS
# ------------------------------------------------------------

for path in TRAIN_FILES:

    print("\n" + "=" * 70)
    print("Loading:", path.name)
    print("=" * 70)

    df = pd.read_csv(
        path,
        low_memory=False
    )

    print("Original rows:", len(df))

    # ----------------------------------------
    # Detect target column
    # ----------------------------------------

    lc = label_col(df)

    print("Label column:", lc)

    # ----------------------------------------
    # Clean infinities
    # ----------------------------------------

    df = df.replace(
        [np.inf, -np.inf],
        np.nan
    )

    # ----------------------------------------
    # Create family labels BEFORE sampling
    # ----------------------------------------

    df["__family_target__"] = (
        df[lc].map(label_to_family)
    )

    df["__binary_target__"] = (
        df["__family_target__"] != "BENIGN"
    ).astype(int)

    # ----------------------------------------
    # Source group
    # ----------------------------------------

    df["__source_group__"] = path.name

    # ----------------------------------------
    # Print original distribution
    # ----------------------------------------

    print("\nOriginal family distribution:")

    print(
        df["__family_target__"]
        .value_counts()
        .to_string()
    )

    # --------------------------------------------------------
    # 11. DETERMINISTIC PER-FILE SAMPLING
    # --------------------------------------------------------
    #
    # IMPORTANT:
    # Do not simply call:
    #
    # df.sample(100000)
    #
    # because that can accidentally eliminate rare attack
    # classes such as Heartbleed or Infiltration.
    #
    # Instead, sample proportionally within each family.
    # --------------------------------------------------------

    if len(df) > TRAIN_PER_FILE_MAX:

        families = []

        family_counts = (
            df["__family_target__"]
            .value_counts()
        )

        # Allocate at least one row to every family.
        number_of_families = len(family_counts)

        remaining_budget = (
            TRAIN_PER_FILE_MAX
            - number_of_families
        )

        if remaining_budget < 0:
            raise ValueError(
                "TRAIN_PER_FILE_MAX is smaller than "
                "the number of families in the file."
            )

        # Start with one row from every family.
        for family in sorted(family_counts.index):

            family_df = df[
                df["__family_target__"] == family
            ]

            sample_n = 1

            sampled = family_df.sample(
                n=sample_n,
                random_state=RANDOM_STATE
            )

            families.append(sampled)

        reserved = pd.concat(
            families,
            ignore_index=False
        )

        remaining = df.drop(
            reserved.index
        )

        # Proportional allocation for remaining rows.
        sampled_parts = [reserved]

        if len(remaining) > 0 and remaining_budget > 0:

            # Calculate family proportions.
            remaining_counts = (
                remaining["__family_target__"]
                .value_counts()
            )

            proportions = (
                remaining_counts
                / remaining_counts.sum()
            )

            allocations = (
                proportions
                * remaining_budget
            ).astype(int)

            # Make sure allocation does not exceed available rows.
            allocations = allocations.clip(
                upper=remaining_counts
            )

            # Distribute rounding remainder.
            allocated = int(
                allocations.sum()
            )

            remainder = (
                remaining_budget
                - allocated
            )

            if remainder > 0:

                candidates = (
                    remaining_counts
                    - allocations
                )

                candidates = (
                    candidates[candidates > 0]
                    .sort_values(ascending=False)
                )

                for family in candidates.index:

                    if remainder <= 0:
                        break

                    allocations.loc[family] += 1
                    remainder -= 1

            for family, n in allocations.items():

                if n <= 0:
                    continue

                family_df = remaining[
                    remaining["__family_target__"] == family
                ]

                sampled = family_df.sample(
                    n=min(n, len(family_df)),
                    random_state=RANDOM_STATE
                )

                sampled_parts.append(sampled)

        df = pd.concat(
            sampled_parts,
            ignore_index=True
        )

        # Shuffle final sampled file deterministically.
        df = df.sample(
            frac=1.0,
            random_state=RANDOM_STATE
        ).reset_index(drop=True)

        print(
            "\nRows after family-aware sampling:",
            len(df)
        )

    else:

        print(
            "\nFile below sampling limit; "
            "all rows retained."
        )

    # ----------------------------------------
    # Append training group
    # ----------------------------------------

    train_parts.append(df)

    print("\nFinal family distribution for this group:")

    print(
        df["__family_target__"]
        .value_counts()
        .to_string()
    )


# ============================================================
# 12. COMBINE TRAINING GROUPS
# ============================================================

if not train_parts:
    raise RuntimeError(
        "No training data was loaded. "
        "Check RAW_DIR and HOLDOUT configuration."
    )

train = pd.concat(
    train_parts,
    ignore_index=True
)

# ============================================================
# 13. FINAL TRAINING DATA REPORT
# ============================================================

print("\n")
print("=" * 70)
print("MODEL 3 V2 GROUP-AWARE TRAINING DATA")
print("=" * 70)

print("\nTraining rows:")
print(len(train))

print("\nTraining groups:")
print(
    train["__source_group__"]
    .nunique()
)

print("\nTraining group names:")

for group in sorted(
    train["__source_group__"].unique()
):
    print(" -", group)

print("\nBinary distribution:")

print(
    train["__binary_target__"]
    .value_counts()
    .sort_index()
)

print("\nBinary distribution (%):")

print(
    (
        train["__binary_target__"]
        .value_counts(normalize=True)
        * 100
    )
    .sort_index()
    .round(3)
)

print("\nAttack-family distribution:")

print(
    train["__family_target__"]
    .value_counts()
    .to_string()
)

# ============================================================
# 14. FINAL HOLDOUT SAFETY CHECK
# ============================================================

training_groups = set(
    train["__source_group__"].unique()
)

overlap = (
    training_groups
    .intersection(HOLDOUT)
)

print("\n")
print("=" * 70)
print("GROUP LEAKAGE CHECK")
print("=" * 70)

print("Training groups:", len(training_groups))
print("Holdout groups:", len(HOLDOUT))

if overlap:

    print(
        "\nERROR: HOLDOUT LEAKAGE DETECTED!"
    )

    for group in sorted(overlap):
        print(" -", group)

    raise RuntimeError(
        "One or more holdout groups entered the training dataset."
    )

else:

    print(
        "\nPASS: No holdout group appears in training data."
    )

# ============================================================
# 15. SAVE PROCESSED TRAINING DATA
# ============================================================

PROCESSED_DIR = (
    PROJECT_DIR
    / "data"
    / "processed"
    / "behavior"
)

PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

TRAIN_CACHE = (
    PROCESSED_DIR
    / "cicids2017_model3_v2_group_aware.parquet"
)

train.to_parquet(
    TRAIN_CACHE,
    index=False
)

print("\nSaved training cache:")
print(TRAIN_CACHE)

print("\nMODEL 3 V2 DATA PREPARATION COMPLETE.")

Project directory:
C:\Users\bimal\OneDrive\Desktop\EVIDENT

Raw behaviour directory:
C:\Users\bimal\OneDrive\Desktop\EVIDENT\data\raw\behaviour

CSV files discovered: 8
 - Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
 - Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
 - Friday-WorkingHours-Morning.pcap_ISCX.csv
 - Monday-WorkingHours.pcap_ISCX.csv
 - Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
 - Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
 - Tuesday-WorkingHours.pcap_ISCX.csv
 - Wednesday-workingHours.pcap_ISCX.csv

Group-aware holdouts:
 - Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
 - Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
 - Friday-WorkingHours-Morning.pcap_ISCX.csv
 - Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv

Training groups:
 - Monday-WorkingHours.pcap_ISCX.csv
 - Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
 - Tuesday-WorkingHours.pcap_ISCX.csv
 - Wednesday-workingHours.pcap_ISCX.csv

Number of t

In [8]:
# Establish numeric feature schema using training data only.
EXCLUDE={"__source_group__","__family_target__","__binary_target__"}
feature_cols=train.select_dtypes(include=[np.number]).columns.tolist()
feature_cols=[c for c in feature_cols if c not in EXCLUDE]
if not feature_cols: raise ValueError("No numeric feature columns found")
X=train[feature_cols].replace([np.inf,-np.inf],np.nan)
y_bin=train["__binary_target__"].astype(int)
print("Feature count:",len(feature_cols))

Feature count: 78


In [9]:
def make_model():
    return Pipeline([
        ("imputer",SimpleImputer(strategy="median")),
        ("scaler",StandardScaler()),
        ("model",HistGradientBoostingClassifier(
            max_iter=350, learning_rate=0.06, max_leaf_nodes=63,
            l2_regularization=1.0, random_state=RANDOM_STATE
        ))
    ])

# Binary training with balanced sample weights
binary_model=make_model()
w_bin=compute_sample_weight(class_weight="balanced",y=y_bin)
binary_model.fit(X,y_bin,model__sample_weight=w_bin)
print("Binary model trained.")

Binary model trained.


In [15]:
# Family model: only attacks, only labels present in training.
attack=train[y_bin==1]
X_f=attack[feature_cols].replace([np.inf,-np.inf],np.nan)
y_f=attack["__family_target__"].astype(str)
print("Known attack families:",sorted(y_f.unique()))
family_model=make_model()
w_f=compute_sample_weight(class_weight="balanced",y=y_f)
family_model.fit(X_f,y_f,model__sample_weight=w_f)
print("Family model trained.")

Known attack families: ['BRUTE_FORCE', 'DOS_DDOS', 'HEARTBLEED', 'WEB_ATTACK']
Family model trained.


In [17]:
# ============================================================
# MODEL 3 V2 — SAVE TRAINED ARTIFACT
# ============================================================

from pathlib import Path
import json
import joblib

# ------------------------------------------------------------
# 1. Existing project structure
# ------------------------------------------------------------

PROJECT_DIR = Path(r"C:\Users\bimal\OneDrive\Desktop\EVIDENT")

BEHAVIOR_DIR = PROJECT_DIR / "models" / "behavior"

ENV_DIR = BEHAVIOR_DIR / "environments"
EXPERIMENT_DIR = BEHAVIOR_DIR / "experiments"
META_DIR = BEHAVIOR_DIR / "metadata"
PRODUCTION_DIR = BEHAVIOR_DIR / "production"

for folder in [
    ENV_DIR,
    EXPERIMENT_DIR,
    META_DIR,
    PRODUCTION_DIR
]:
    folder.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# 2. Check that training objects exist
# ------------------------------------------------------------

print("=" * 70)
print("CHECKING TRAINING OBJECTS")
print("=" * 70)

print("binary_model :", type(binary_model).__name__)
print("family_model :", type(family_model).__name__)
print("feature_cols :", len(feature_cols))
print("training rows:", len(train))
print("holdout groups:", len(HOLDOUT))


# ------------------------------------------------------------
# 3. Training families
# ------------------------------------------------------------

known_families = sorted(
    [str(x) for x in y_f.dropna().unique()]
)

print("\nKnown families learned by family model:")
for family in known_families:
    print(" -", family)


# ------------------------------------------------------------
# 4. Build Model 3 V2 bundle
# ------------------------------------------------------------

bundle = {
    "version": "2.0.0-group-aware",

    "dataset": "CIC-IDS2017",

    "feature_columns": list(feature_cols),

    "binary_model": binary_model,

    "family_model": family_model,

    "binary_threshold": 0.5,

    "family_labels": known_families,

    "training_groups": sorted(
        [
            str(x)
            for x in train["__source_group__"].unique()
        ]
    ),

    "holdout_groups": [
        str(x) for x in HOLDOUT
    ],

    "training_rows": int(len(train)),

    "random_state": int(RANDOM_STATE),

    "model_type": "HistGradientBoostingClassifier",

    "sampling_per_training_file_max": int(
        TRAIN_PER_FILE_MAX
    )
}


# ------------------------------------------------------------
# 5. Save V2 experiment artifact
# ------------------------------------------------------------

MODEL_PATH = (
    EXPERIMENT_DIR /
    "network_threat_model_v2_group_aware.pkl"
)

joblib.dump(
    bundle,
    MODEL_PATH
)

print("\nModel saved successfully:")
print(MODEL_PATH)


# ------------------------------------------------------------
# 6. Create metadata
# ------------------------------------------------------------

metadata = {
    "version": "2.0.0-group-aware",

    "dataset": "CIC-IDS2017",

    "artifact": str(MODEL_PATH),

    "training_rows": int(len(train)),

    "training_groups": bundle[
        "training_groups"
    ],

    "holdout_groups": bundle[
        "holdout_groups"
    ],

    "feature_count": int(
        len(feature_cols)
    ),

    "known_families": known_families,

    "binary_threshold": 0.5,

    "model_type": (
        "HistGradientBoostingClassifier"
    ),

    "selection": (
        "higher_capacity architecture "
        "with balanced sample weights"
    ),

    "random_state": int(RANDOM_STATE),

    "sampling_per_training_file_max": int(
        TRAIN_PER_FILE_MAX
    ),

    "status": "EXPERIMENTAL_CANDIDATE",

    "production_promoted": False
}


# ------------------------------------------------------------
# 7. Save metadata
# ------------------------------------------------------------

METADATA_PATH = (
    META_DIR /
    "model3_v2_training_metadata.json"
)

with open(
    METADATA_PATH,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        metadata,
        f,
        indent=2
    )

print("Metadata saved successfully:")
print(METADATA_PATH)


# ------------------------------------------------------------
# 8. Final verification
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("MODEL 3 V2 SAVING COMPLETE")
print("=" * 70)

print("\nMODEL:")
print(MODEL_PATH)

print("\nMETADATA:")
print(METADATA_PATH)

print("\nKnown families:")
print(known_families)

print("\nTraining groups:")
for group in bundle["training_groups"]:
    print(" -", group)

print("\nHoldout groups:")
for group in bundle["holdout_groups"]:
    print(" -", group)

print("\nStatus:")
print("EXPERIMENTAL_CANDIDATE")

print("\nProduction model:")
print("NOT MODIFIED")

print("\n" + "=" * 70)

CHECKING TRAINING OBJECTS
binary_model : Pipeline
family_model : Pipeline
feature_cols : 78
training rows: 400000
holdout groups: 4

Known families learned by family model:
 - BRUTE_FORCE
 - DOS_DDOS
 - HEARTBLEED
 - WEB_ATTACK

Model saved successfully:
C:\Users\bimal\OneDrive\Desktop\EVIDENT\models\behavior\experiments\network_threat_model_v2_group_aware.pkl
Metadata saved successfully:
C:\Users\bimal\OneDrive\Desktop\EVIDENT\models\behavior\metadata\model3_v2_training_metadata.json

MODEL 3 V2 SAVING COMPLETE

MODEL:
C:\Users\bimal\OneDrive\Desktop\EVIDENT\models\behavior\experiments\network_threat_model_v2_group_aware.pkl

METADATA:
C:\Users\bimal\OneDrive\Desktop\EVIDENT\models\behavior\metadata\model3_v2_training_metadata.json

Known families:
['BRUTE_FORCE', 'DOS_DDOS', 'HEARTBLEED', 'WEB_ATTACK']

Training groups:
 - Monday-WorkingHours.pcap_ISCX.csv
 - Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
 - Tuesday-WorkingHours.pcap_ISCX.csv
 - Wednesday-workingHours.pcap_IS